In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')
import workflow_objects as wfo
import logging_config as lcf
import database_interact as dbi
import load_tools as ldt

logger = lcf.get_logger('workflow_example')

In [2]:
# def run_targeted_metabolomics_project(config: dict, project_db_path: str, atlas_uids: dict) -> None:
#     """
#     Run the complete targeted metabolomics workflow starting from pre-existing atlases.
    
#     Args:
#         config: Configuration dictionary
#         project_db_path: Path to project DuckDB database (contains H5 files and will store results)
#         atlas_uids: Dictionary mapping atlas types to UIDs, e.g.:
#                    {
#                        'qc': {'hilic_positive': 'qc_atlas_uid_123', 'hilic_negative': 'qc_atlas_uid_456'},
#                        'istd': {'hilic_positive': 'istd_atlas_uid_789'},
#                        'ema': {'hilic_positive': 'ema_atlas_uid_101112'}
#                    }
#     """
    
#     main_db_path = config["paths"]["main_database"]
#     project_directory = str(Path(project_db_path).parent)  # Derive from project DB path
    
#     # Initialize workflow orchestrator with atlas UIDs
#     workflow = wfo.TargetedMetabolomicsWorkflow(
#         config=config,
#         project_db_path=project_db_path,
#         project_directory=project_directory,
#         main_db_path=main_db_path,
#         atlas_uids=atlas_uids  # Pass the pre-existing atlas UIDs
#     )
    
#     logger.info("Starting targeted metabolomics workflow...")
#     logger.info(f"Using atlas UIDs: {atlas_uids}")
    
#     # Run workflow up to manual curation (returns GUI)
#     gui = workflow.run_complete_workflow(stop_at_stage=wfo.WorkflowStage.MANUAL_CURATION)
    
#     # Display status
#     status = workflow.get_workflow_status()
#     logger.info(f"Workflow status: {status}")
    
#     # The GUI is returned for manual curation
#     if gui:
#         logger.info("Manual curation GUI is ready. Complete your annotations and then run:")
#         logger.info("final_report = workflow.continue_to_final_report()")
#         return gui
    
#     return workflow

# def run_workflow_by_atlas_type(config: dict, project_db_path: str, atlas_uid: str, 
#                               atlas_type: str = 'qc') -> None:
#     """
#     Run workflow for a specific atlas UID and type.
    
#     Args:
#         config: Configuration dictionary
#         project_db_path: Path to project DuckDB database
#         atlas_uid: Specific atlas UID to process
#         atlas_type: Type of atlas ('qc', 'istd', 'ema')
#     """
    
#     main_db_path = config["paths"]["main_database"]
#     project_directory = str(Path(project_db_path).parent)
    
#     # Create single-atlas dictionary
#     atlas_uids = {atlas_type: {'single_method': atlas_uid}}
    
#     # Initialize and run through putative identification
#     workflow = wfo.TargetedMetabolomicsWorkflow(
#         config=config,
#         project_db_path=project_db_path,
#         project_directory=project_directory,
#         main_db_path=main_db_path,
#         atlas_uids=atlas_uids
#     )
    
#     # Run up to putative identification
#     workflow.run_complete_workflow(stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION)
    
#     # Create GUI for specific atlas type
#     curation_manager = wfo.ManualCurationManager(workflow.putative_identification)
#     gui = curation_manager.create_curation_gui(config, atlas_type=atlas_type)
    
#     logger.info(f"Created curation GUI for {atlas_type} atlas type")
#     return gui

# def create_atlas_uids_from_project_atlases(project_db_path: str) -> dict:
#     """
#     Helper function to extract atlas UIDs from project database.
#     Use this if atlases are already stored in the project database.
    
#     Returns:
#         Dictionary of atlas UIDs organized by type and method
#     """
#     atlas_uids = {'qc': {}, 'istd': {}, 'ema': {}}
    
#     # Query project database for existing atlases
#     with dbi.get_db_connection(project_db_path) as conn:
#         results = conn.execute("""
#             SELECT atlas_uid, atlas_name, atlas_type, chromatography, polarity
#             FROM atlases 
#             WHERE atlas_type IN ('qc', 'istd', 'ema')
#             ORDER BY atlas_type, chromatography, polarity
#         """).fetchall()
    
#     for atlas_uid, atlas_name, atlas_type, chromatography, polarity in results:
#         chrom_pol_key = f"{chromatography}_{polarity}"
#         atlas_uids[atlas_type][chrom_pol_key] = atlas_uid
#         logger.info(f"Found {atlas_type} atlas: {atlas_name} ({chrom_pol_key}) -> {atlas_uid}")
    
#     return atlas_uids

# def inspect_putative_identifications(project_db_path: str) -> None:
#     """
#     Example of inspecting putative identifications without running the full workflow.
#     """
    
#     # Get all targeted analysis results
#     with dbi.get_db_connection(project_db_path) as conn:
#         results = conn.execute("""
#             SELECT DISTINCT atlas_type, chromatography_polarity, 
#                    COUNT(*) as compound_count,
#                    SUM(CASE WHEN curation_status = 'finalized' THEN 1 ELSE 0 END) as finalized_count
#             FROM targeted_analysis 
#             GROUP BY atlas_type, chromatography_polarity
#             ORDER BY atlas_type, chromatography_polarity
#         """).fetchall()
    
#     logger.info("Putative identification summary:")
#     for row in results:
#         atlas_type, chrom_pol, total, finalized = row
#         logger.info(f"  {atlas_type} - {chrom_pol}: {total} compounds ({finalized} finalized)")

# def generate_atlas_type_reports(config: dict, project_db_path: str, output_dir: str) -> None:
#     """
#     Example of generating separate reports for each atlas type.
#     """
    
#     main_db_path = config["paths"]["main_database"]
#     output_path = Path(output_dir)
#     output_path.mkdir(parents=True, exist_ok=True)
    
#     # Get atlas UIDs from project database
#     atlas_uids = create_atlas_uids_from_project_atlases(project_db_path)
    
#     # Load existing workflow state
#     workflow = wfo.TargetedMetabolomicsWorkflow(
#         config=config,
#         project_db_path=project_db_path,
#         project_directory=str(output_path.parent),
#         main_db_path=main_db_path,
#         atlas_uids=atlas_uids
#     )
    
#     # Generate reports for each atlas type
#     atlas_types = ['qc', 'istd', 'ema']
    
#     for atlas_type in atlas_types:
#         logger.info(f"Generating report for {atlas_type} atlas")
        
#         # Create filtered report
#         # This would need to be implemented in the FinalReportManager
#         report_path = output_path / f"{atlas_type}_targeted_analysis_report"
        
#         # Placeholder - would implement filtered report generation
#         logger.info(f"Report saved to {report_path}")

In [3]:
# # Load configuration
# CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")

# # Example project setup
# PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"
# ATLAS_INPUTS = {
#     'qc': {
#         'hilic_positive': 'atl-raw-ISTD-8d812bf22b8a4e42af17b0e77cab6db8',
#     },
#     'istd': {
#         'hilic_positive': 'atl-raw-QC-8733960f481b4b6bb43329c2edf02f4b',
#     },
#     'ema': {
#     }
# }

In [4]:
# Run complete workflow with pre-defined atlas UIDs
# workflow_result = run_targeted_metabolomics_project(
#     CONFIG, 
#     PROJECT_DB_PATH, 
#     ATLAS_INPUTS
# )

# Example 2: Extract atlas UIDs from project database (if they're already stored there)
# atlas_uids_from_db = create_atlas_uids_from_project_atlases(PROJECT_DB_PATH)
# workflow_result = run_targeted_metabolomics_project(CONFIG, PROJECT_DB_PATH, atlas_uids_from_db)

# Example 3: Run for specific atlas UID and type
# single_atlas_uid = 'qc_hilic_pos_atlas_uid_123'
# qc_gui = run_workflow_by_atlas_type(CONFIG, PROJECT_DB_PATH, single_atlas_uid, atlas_type='qc')

# Example 4: Inspect existing results
# inspect_putative_identifications(PROJECT_DB_PATH)

In [ ]:
def complete_workflow_example(config: dict, project_name: str):
    """
    Complete example showing upstream atlas creation + targeted analysis workflow.
    This demonstrates the full pipeline from CSV files to final analysis results.
    """
    
    # =============================================================================
    # STAGE 1: UPSTREAM ATLAS CREATION (Main Database)
    # =============================================================================
    
    logger.info("=== STAGE 1: CREATING ATLASES FROM CSV FILES ===")
    
    input_atlases = config.get('atlases', {})
    print(input_atlases)
    atlas_uids = {}
    
    for atlas_type, atlas_info in input_atlases.items():
        csv_path = atlas_info['atlas_table_path']
        atlas_name = atlas_info.get('atlas_name', 'No name')
        atlas_description = atlas_info.get('atlas_description', 'No description')

        if Path(csv_path).exists():
            logger.info(f"Processing {atlas_type} compounds from {csv_path}")
            
            # Load compounds from CSV
            compounds_df = ldt.load_atlas_input(csv_path)
            atlas_chromatography = ldt.detect_atlas_input_chromatography(compounds_df)
            atlas_polarity = ldt.detect_atlas_input_polarity(compounds_df)
            logger.info(f"  Loaded {len(compounds_df)} compounds")
            
            # Add compounds to main database (creates compounds + mz_rt_references)
            dbi.add_compounds_to_db(compounds_df, config, csv_path)
            
            # Create atlas from compounds (creates atlas + associations)
            atlas_uid, atlas_name = dbi.create_atlas_from_compounds(
                compounds_df,
                atlas_name,
                atlas_description,
                atlas_type,
                atlas_chromatography,
                atlas_polarity,
                config
            )
            
            # Store atlas UID for targeted analysis
            chrom_pol_key = f"{atlas_chromatography}_{atlas_polarity}"
            if atlas_type not in atlas_uids:
                atlas_uids[atlas_type] = {}
            atlas_uids[atlas_type][chrom_pol_key] = atlas_uid
            
            logger.info(f"  Created atlas: {atlas_name} (UID: {atlas_uid})")
        else:
            logger.warning(f"CSV file not found: {csv_path}")
    
    logger.info(f"Atlas creation complete. Created {sum(len(v) for v in atlas_uids.values())} atlases")
    logger.info(f"Atlas UIDs: {atlas_uids}")
    
    # =============================================================================
    # STAGE 2: TARGETED ANALYSIS WORKFLOW (Project Database)
    # =============================================================================
    
    logger.info("=== STAGE 2: TARGETED ANALYSIS WORKFLOW ===")
    
    # Setup project paths
    project_directory = f"{config['paths']['projects_dir']}/{project_name}"
    project_db_path = f"{project_directory}/{project_name}.duckdb"
    
    # Ensure project directory exists
    Path(project_directory).mkdir(parents=True, exist_ok=True)
    
    # Initialize targeted metabolomics workflow
    workflow = wfo.TargetedMetabolomicsWorkflow(
        config=config,
        project_db_path=project_db_path,
        project_directory=project_directory,
        main_db_path=config["paths"]["main_database"],
        atlas_uids=atlas_uids  # Use atlas UIDs from Stage 1
    )
    
    # Run complete workflow
    try:
        logger.info("Starting targeted analysis workflow...")
        result = workflow.run_complete_workflow(stop_at_stage=wfo.WorkflowStage.MANUAL_CURATION)
        
        if result:  # GUI returned for manual curation
            logger.info("Workflow reached manual curation stage")
            logger.info("GUI is ready for manual compound annotation")
            logger.info("After manual curation, run: workflow.continue_to_final_report()")
            return result, workflow
        else:
            logger.info("Workflow completed successfully")
            return None, workflow
            
    except Exception as e:
        logger.error(f"Workflow failed: {e}")
        raise
    
    return workflow

In [14]:
# Load configuration
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")

# Run complete workflow from CSV to analysis
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
gui, workflow = complete_workflow_example(CONFIG, PROJECT_NAME)

2025-09-09 13:32:24 | metatlas2.workflow_example | INFO | === STAGE 1: CREATING ATLASES FROM CSV FILES ===
{'qc': {'atlas_table_path': '/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv', 'atlas_name': 'HILICZ QC Atlas Positive', 'atlas_description': 'Default QC compounds for HILIC positive mode analysis'}, 'istd': {'atlas_table_path': '/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv', 'atlas_name': 'HILICZ ISTD Atlas Positive', 'atlas_description': 'Default internal standard compounds for HILIC positive mode analysis'}, 'ema': {'atlas_table_path': '/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv', 'atlas_name': 'HILICZ EMA Atlas Positive', 'atlas_description': 'Experimental compounds for HILIC positive mode analysis'}}
2025-09-09 13:32:24 | metatlas2.workflow_example | INFO | Processing qc compounds from /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv
2025-0

Preparing compounds:   0%|          | 0/20 [00:00<?, ?it/s]

2025-09-09 13:32:24 | metatlas2.database_interact | INFO | Checking for duplicate references and batch inserting...
2025-09-09 13:32:24 | metatlas2.database_interact | INFO | Compounds added successfully!
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Total compounds in database: 482
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    New compounds created: 0
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Compounds skipped (already existed): 20
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Total RT/MZ references in database: 860
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    New RT/MZ references created: 0
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    RT/MZ references skipped (duplicates): 20
2025-09-09 13:32:24 | metatlas2.database_interact | INFO | Created atlas 'HILICZ QC Atlas Positive' with UID: atl-raw-qc-11f44bd079d04d68ab7b106d20279712
2025-09-09 13:32:24 | metatlas2.database_interact | INFO 

Preparing compounds:   0%|          | 0/65 [00:00<?, ?it/s]

2025-09-09 13:32:24 | metatlas2.database_interact | INFO | Checking for duplicate references and batch inserting...
2025-09-09 13:32:24 | metatlas2.database_interact | INFO | Compounds added successfully!
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Total compounds in database: 482
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    New compounds created: 0
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Compounds skipped (already existed): 65
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Total RT/MZ references in database: 860
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    New RT/MZ references created: 0
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    RT/MZ references skipped (duplicates): 65
2025-09-09 13:32:24 | metatlas2.database_interact | INFO | Created atlas 'HILICZ ISTD Atlas Positive' with UID: atl-raw-istd-99fe687f17094c78bd004c43a7a9b315
2025-09-09 13:32:24 | metatlas2.database_interact | I

Preparing compounds:   0%|          | 0/373 [00:00<?, ?it/s]

2025-09-09 13:32:24 | metatlas2.database_interact | INFO | Checking for duplicate references and batch inserting...
2025-09-09 13:32:24 | metatlas2.database_interact | INFO | Compounds added successfully!
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Total compounds in database: 482
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    New compounds created: 0
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Compounds skipped (already existed): 373
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    Total RT/MZ references in database: 860
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    New RT/MZ references created: 0
2025-09-09 13:32:24 | metatlas2.database_interact | INFO |    RT/MZ references skipped (duplicates): 373
2025-09-09 13:32:25 | metatlas2.database_interact | INFO | Created atlas 'HILICZ EMA Atlas Positive' with UID: atl-raw-ema-04f4e429d54742e39c3a7a137de7c751
2025-09-09 13:32:25 | metatlas2.database_interact | I

In [ ]:
# def query_available_atlases(config: dict, atlas_type: str = None) -> pd.DataFrame:
#     """
#     Query available atlases from main database.
#     Useful for finding existing atlas UIDs for targeted analysis.
    
#     Args:
#         config: Configuration dictionary
#         atlas_type: Filter by atlas type ('qc', 'istd', 'ema') or None for all
    
#     Returns:
#         DataFrame with atlas information
#     """
#     main_db_path = config["paths"]["main_database"]
    
#     with dbi.get_db_connection(main_db_path) as conn:
#         if atlas_type:
#             query = """
#                 SELECT 
#                     atlas_uid,
#                     atlas_name,
#                     atlas_description,
#                     atlas_type,
#                     chromatography,
#                     polarity,
#                     created_by,
#                     last_modified,
#                     (SELECT COUNT(*) 
#                      FROM atlas_compound_associations aca 
#                      WHERE aca.atlas_uid = a.atlas_uid) as compound_count
#                 FROM atlases a
#                 WHERE atlas_type = ?
#                 ORDER BY atlas_type, chromatography, polarity, atlas_name
#             """
#             params = [atlas_type]
#         else:
#             query = """
#                 SELECT 
#                     atlas_uid,
#                     atlas_name, 
#                     atlas_description,
#                     atlas_type,
#                     chromatography,
#                     polarity,
#                     created_by,
#                     last_modified,
#                     (SELECT COUNT(*) 
#                      FROM atlas_compound_associations aca 
#                      WHERE aca.atlas_uid = a.atlas_uid) as compound_count
#                 FROM atlases a
#                 ORDER BY atlas_type, chromatography, polarity, atlas_name
#             """
#             params = []
        
#         df = conn.execute(query, params).df()
    
#     if df.empty:
#         logger.info(f"No atlases found" + (f" for type '{atlas_type}'" if atlas_type else ""))
#     else:
#         logger.info(f"Found {len(df)} atlases" + (f" for type '{atlas_type}'" if atlas_type else ""))
    
#     return df

# def build_atlas_uids_from_query(config: dict, filter_criteria: dict = None) -> dict:
#     """
#     Build atlas_uids dictionary from database query with optional filtering.
    
#     Args:
#         config: Configuration dictionary
#         filter_criteria: Optional dict to filter atlases, e.g.:
#                         {'chromatography': 'hilic', 'polarity': 'positive'}
    
#     Returns:
#         Dictionary in format expected by TargetedMetabolomicsWorkflow
#     """
#     available_atlases = query_available_atlases(config)
    
#     if available_atlases.empty:
#         logger.warning("No atlases found in database")
#         return {'qc': {}, 'istd': {}, 'ema': {}}
    
#     # Apply filters if provided
#     if filter_criteria:
#         for key, value in filter_criteria.items():
#             if key in available_atlases.columns:
#                 available_atlases = available_atlases[available_atlases[key] == value]
#                 logger.info(f"Filtered by {key}='{value}': {len(available_atlases)} atlases remaining")
    
#     # Build atlas_uids dictionary
#     atlas_uids = {'qc': {}, 'istd': {}, 'ema': {}}
    
#     for _, row in available_atlases.iterrows():
#         atlas_type = row['atlas_type']
#         if atlas_type in atlas_uids:
#             chrom_pol_key = f"{row['chromatography']}_{row['polarity']}"
#             atlas_uids[atlas_type][chrom_pol_key] = row['atlas_uid']
            
#             logger.info(f"Added {atlas_type} atlas: {row['atlas_name']} -> {chrom_pol_key}")
    
#     return atlas_uids

# def display_atlas_summary(config: dict):
#     """
#     Display a nice summary of available atlases grouped by type and method.
#     """
#     available_atlases = query_available_atlases(config)
    
#     if available_atlases.empty:
#         print("No atlases found in database")
#         return
    
#     print("Available Atlases Summary:")
#     print("=" * 50)
    
#     # Group by atlas type
#     for atlas_type in ['qc', 'istd', 'ema']:
#         type_atlases = available_atlases[available_atlases['atlas_type'] == atlas_type]
#         if not type_atlases.empty:
#             print(f"\n{atlas_type.upper()} Atlases ({len(type_atlases)}):")
#             print("-" * 30)
            
#             for _, row in type_atlases.iterrows():
#                 method = f"{row['chromatography']}_{row['polarity']}"
#                 print(f"  {method:20} | {row['atlas_name']:40} | {row['compound_count']:3d} compounds | {row['atlas_uid']}")

# # Utility functions for working with existing atlases
# def get_atlas_details(config: dict, atlas_uid: str) -> dict:
#     """Get detailed information about a specific atlas."""
#     main_db_path = config["paths"]["main_database"]
    
#     with dbi.get_db_connection(main_db_path) as conn:
#         # Get atlas metadata
#         atlas_query = """
#             SELECT * FROM atlases WHERE atlas_uid = ?
#         """
#         atlas_info = conn.execute(atlas_query, [atlas_uid]).fetchone()
        
#         if not atlas_info:
#             logger.error(f"Atlas {atlas_uid} not found")
#             return {}
        
#         # Get compound count and sample compounds
#         compounds_query = """
#             SELECT 
#                 c.compound_name,
#                 c.inchi_key,
#                 c.formula,
#                 mzrt.mz,
#                 mzrt.rt_peak,
#                 mzrt.adduct
#             FROM atlas_compound_associations aca
#             JOIN compounds c ON aca.compound_uid = c.compound_uid
#             JOIN mz_rt_references mzrt ON aca.mz_rt_reference_uid = mzrt.mz_rt_reference_uid
#             WHERE aca.atlas_uid = ?
#             ORDER BY mzrt.rt_peak
#         """
#         compounds_df = conn.execute(compounds_query, [atlas_uid]).df()
    
#     return {
#         'atlas_info': dict(zip([desc[0] for desc in conn.description], atlas_info)),
#         'compound_count': len(compounds_df),
#         'compounds_sample': compounds_df.head(10),  # First 10 compounds
#         'rt_range': (compounds_df['rt_peak'].min(), compounds_df['rt_peak'].max()) if not compounds_df.empty else (None, None)
#     }

# # Example usage for querying existing atlases
# def example_query_existing_atlases():
#     """Example of how to query and use existing atlases for targeted analysis."""
    
#     # Load configuration
#     CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
    
#     # Display all available atlases
#     print("=== ALL AVAILABLE ATLASES ===")
#     display_atlas_summary(CONFIG)
    
#     # Query specific atlas types
#     print("\n=== QC ATLASES ===")
#     qc_atlases = query_available_atlases(CONFIG, 'qc')
#     print(qc_atlases[['atlas_name', 'chromatography', 'polarity', 'compound_count', 'atlas_uid']])
    
#     # Build atlas UIDs for HILIC positive only
#     atlas_uids = build_atlas_uids_from_query(CONFIG, {
#         'chromatography': 'hilic',
#         'polarity': 'positive'
#     })
    
#     print(f"\n=== HILIC POSITIVE ATLAS UIDs ===")
#     print(f"Atlas UIDs for targeted analysis: {atlas_uids}")
    
#     # If we have atlases, run targeted analysis
#     if any(atlas_uids.values()):
#         PROJECT_NAME = "example_from_existing_atlases"
#         project_directory = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}"
#         project_db_path = f"{project_directory}/{PROJECT_NAME}.duckdb"
        
#         print(f"\n=== RUNNING TARGETED ANALYSIS ===")
#         print(f"Using existing atlas UIDs: {atlas_uids}")
        
#         # Run workflow with existing atlases
#         workflow_result = run_targeted_metabolomics_project(
#             CONFIG, 
#             project_db_path, 
#             atlas_uids
#         )
        
#         return workflow_result
#     else:
#         print("No matching atlases found for analysis")
#         return None

# # Quick atlas lookup by name pattern
# def find_atlases_by_name(config: dict, name_pattern: str) -> pd.DataFrame:
#     """Find atlases by name pattern (case-insensitive)."""
#     available_atlases = query_available_atlases(config)
    
#     if available_atlases.empty:
#         return pd.DataFrame()
    
#     # Filter by name pattern
#     mask = available_atlases['atlas_name'].str.contains(name_pattern, case=False, na=False)
#     matches = available_atlases[mask]
    
#     logger.info(f"Found {len(matches)} atlases matching pattern '{name_pattern}'")
    
#     return matches